In [61]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta as _td
from google.auth import aws
from google.cloud import bigquery
from google.api_core import exceptions as gcp_exc


In [62]:

IMDS = "http://169.254.169.254/latest"

def _imds_token():
    return requests.put(
        f"{IMDS}/api/token",
        headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
        timeout=2,
    ).text

class IMDSv2Supplier(aws.AwsSecurityCredentialsSupplier):
    def get_aws_region(self, context, request):
        az = requests.get(
            f"{IMDS}/meta-data/placement/availability-zone",
            headers={"X-aws-ec2-metadata-token": _imds_token()},
            timeout=2,
        ).text
        return az[:-1]

    def get_aws_security_credentials(self, context, request):
        tok = _imds_token()
        role = requests.get(
            f"{IMDS}/meta-data/iam/security-credentials/",
            headers={"X-aws-ec2-metadata-token": tok},
            timeout=2,
        ).text.strip()
        creds = json.loads(requests.get(
            f"{IMDS}/meta-data/iam/security-credentials/{role}",
            headers={"X-aws-ec2-metadata-token": tok},
            timeout=2,
        ).text)
        return aws.AwsSecurityCredentials(
            access_key_id=creds["AccessKeyId"],
            secret_access_key=creds["SecretAccessKey"],
            session_token=creds.get("Token"),
        )

with open("/home/ubuntu/project/final_credentials.json") as f:
    info = json.load(f)

credentials = aws.Credentials(
    audience=info["audience"],
    subject_token_type=info["subject_token_type"],
    token_url=info["token_url"],
    service_account_impersonation_url=info["service_account_impersonation_url"],
    aws_security_credentials_supplier=IMDSv2Supplier(),
)

client = bigquery.Client(project="analog-subset-491623-j7", credentials=credentials)

query = "SELECT 'Handshake Successful!' as status"
for row in client.query(query):
    print(row.status)

Handshake Successful!


In [63]:
# Flash Loan Cascade: Deep Sequence Detection
# Goal: Detect complex multi-protocol arbitrage and exploit chains using "No-Join" account arrays.
# Ideal for: Transformer AE "Feature-Attention" and Sequence Profiling.
# ── UPDATED CONFIGURATION FOR CASCADES ────────────────────────────
DATASET_ID = 'my_dataset_uc_4'
# Daily chunks: one feather file per day across 5 months
_d, _end = date(2024, 9, 1), date(2025, 1, 31)
chunks = []
while _d <= _end:
    _s = _d.strftime('%Y-%m-%d')
    chunks.append((_s, _s))
    _d += _td(days=1)
names = [c[0] for c in chunks]
# Removed deprecated features from array to match the new SQL query exactly.
EXPECTED_FEATURE_COLUMNS = [
    # Base Transaction Data
    'signature', 'block_timestamp', 'tx_date', 'wallet', 'fee_sol', 'compute_units_consumed',
    'success_flag', 'num_accounts', 'log_count', 'num_balance_changes', 'max_sol_change',

    # Protocol Hop Features
    'dex_hop_count', 'debt_hop_count', 'unique_nonsigner_account_count',

    # Execution Structure
    'involves_sysvar_flag', 'max_cpi_depth', 'inner_instruction_count', 'unique_program_count',

    # Value Extraction
    'instigator_net_token_profit', 'instigator_sol_delta', 'instigator_fiat_delta',
    'mint_diversity',
]
# This defines the numeric subset passed into the Transformer/Scaler
# Engineered features (hop_density, avg_depth_per_protocol, unknown_program_count,
# has_fiat_profit, has_token_profit, has_sol_profit) are added by preprocess.engineer_features()
# at training time -- not present in raw feather files.
tx_feature_cols = [
    'fee_sol', 'compute_units_consumed', 'num_accounts', 'log_count',
    'num_balance_changes', 'max_sol_change', 'dex_hop_count', 'debt_hop_count',
    'unique_nonsigner_account_count',
    'involves_sysvar_flag', 'max_cpi_depth', 'inner_instruction_count',
    'instigator_net_token_profit', 'instigator_sol_delta', 'instigator_fiat_delta',
    'mint_diversity',
]
# ──────────────────────────────────────────────────────────────────


In [64]:
def get_cascade_query(dataset_id, start_date, end_date, daily_cap=None):
    limit_clause = f"ORDER BY block_timestamp, signature LIMIT {daily_cap}" if daily_cap else ""
    return f"""
CREATE OR REPLACE TABLE {dataset_id}.flash_loan_cascade_features
PARTITION BY tx_date
CLUSTER BY wallet
AS
WITH raw_base AS (
  SELECT
    signature,
    block_timestamp,
    DATE(block_timestamp) AS tx_date,
    fee / 1e9 AS fee_sol,
    compute_units_consumed,
    IF(status='Success',1,0) AS success_flag,
    
    -- IDENTIFY PRIMARY ACTOR (Excludes infrastructure & keeper bots)
    (SELECT a.pubkey FROM UNNEST(accounts) a WITH OFFSET pos
     WHERE a.signer 
       AND a.pubkey NOT IN (
          -- System / Infrastructure
          '11111111111111111111111111111111',            -- System Program
          'ComputeBudget111111111111111111111111111111', -- Compute Budget Program
          -- DEX / AMMs
          'JUP6LkbZbjS1jKKwapdHNy74zcZ3tLUZoi5QNyVTaV4', -- Jupiter v6 Aggregator
          'jupPAzUY5Ncvrwp3WbagoQL3wr2gJnK9KWwVMFjgCMm', -- Jupiter v3 Aggregator
          '675kPX9MHTjS2zt1qfr1NYHuzeLXfQM9H24wFSUt1Mp8', -- Raydium AMM v4
          'whirLuv3caEkYvHWfCHsrTfcTGSW7uRckyAm7pKA5MC', -- Orca Whirlpool
          '6EF8rrecthR5Dkzon8Nwu78hRvfCKubJ14M5uBEwF6P', -- Pump.fun
          'LBUZKhR7JkaNmMbhxuCc3C6rSkmcz6ZXLMa7YjtAn2D', -- Meteora DLMM
          'PhoeNiXZ8ByJGLkxNfZRnkUfjvmuYqLR89jjFHGqdXY', -- Phoenix DEX
          '2wT8Yq49kHgDzXuPxZSaeLaH1qbmGXtEyPy64bL7aD3c', -- Lifinity V2
          '9xQeWvG816bUx9EPjHmaT23yvVM2ZWbrrpZb9PusVFin', -- Serum DEX v3 / OpenBook
          'SSwpkEEcbUqx4vtoEByFjSkhKdCT862DNVb52nZg1UZ', -- Saber Stable AMM
          -- Lending / Perps
          '6LtLpnUFNByNXLyCoK9wA2MykKAmQNZKBdY8s47dehDc', -- Kamino Finance
          'So1endDq2YkqTCqM3vYpe61CPLUD6nHRN4um8HEufXq', -- Solend
          'MFv2hWf31Z9kbCa1snEPYpcwiNcNVt7BvR9LpXkXVwN', -- MarginFi
          'dRiftyHA39MWEi3m9aunc5MzRF1JYuBsEX6TUqavNbc', -- Drift Protocol
          '4MangoMjqJ2firMokCjjGgoK8d4MXcrgL7XJaL3w6fVg', -- Mango Markets V4
          'mv3ekLzLbnVPNxjZApeBp42ewPtzy6H8N15Kq6n2Y1z', -- Mango Markets V3
          'ZETAxsqBRek56DhiGXrn75yj2NHU3aYUnxvHXpkf3aD'  -- Zeta Markets
       ) 
     ORDER BY pos ASC
     LIMIT 1) AS wallet,
     
    accounts,
    log_messages,
    pre_token_balances,
    post_token_balances,
    balance_changes
  FROM `bigquery-public-data.crypto_solana_mainnet_us.Transactions`
  WHERE block_timestamp >= TIMESTAMP('{start_date}') AND block_timestamp < TIMESTAMP_ADD(TIMESTAMP('{end_date}'), INTERVAL 1 DAY)
    AND status = 'Success'
    -- TRIGGER: Focus on complex transactions
    AND (compute_units_consumed > 500000 OR ARRAY_LENGTH(log_messages) > 30)
),
stratified_sample AS (
  -- WALLET-LEVEL STRATIFIED SAMPLING: Keep 10% of unique wallets
  SELECT * FROM raw_base
  WHERE MOD(ABS(FARM_FINGERPRINT(wallet)), 10) = 0
)
SELECT
    signature,
    block_timestamp,
    tx_date,
    wallet,
    fee_sol,
    compute_units_consumed,
    success_flag,
    -- 1. THE FLASH LOAN SIGNAL (Instructions Sysvar)
    (SELECT IF(COUNT(1) > 0, 1, 0) FROM UNNEST(accounts) a 
     WHERE a.pubkey = 'Sysvar1nstructions1111111111111111111111111') AS involves_sysvar_flag,
    -- 2. CASCADE LOGIC (Depth & Count)
    (SELECT MAX(CAST(REGEXP_EXTRACT(msg, r'invoke \\[(\\d+)\\]') AS INT64)) FROM UNNEST(log_messages) msg) AS max_cpi_depth,
    (SELECT COUNTIF(REGEXP_CONTAINS(msg, r'Program .* invoke')) FROM UNNEST(log_messages) msg) AS inner_instruction_count,
    -- 3a. DEX TRANSIT COUNT (Aggregators / AMMs)
    (SELECT COUNT(DISTINCT a.pubkey) FROM UNNEST(accounts) a 
     WHERE a.pubkey IN (
        'JUP6LkbZbjS1jKKwapdHNy74zcZ3tLUZoi5QNyVTaV4', -- Jupiter
        '675kPX9MHTjS2zt1qfr1NYHuzeLXfQM9H24wFSUt1Mp8', -- Raydium
        'whirLuv3caEkYvHWfCHsrTfcTGSW7uRckyAm7pKA5MC', -- Orca Whirlpool
        '6EF8rrecthR5Dkzon8Nwu78hRvfCKubJ14M5uBEwF6P', -- Pump.fun
        'LBUZKhR7JkaNmMbhxuCc3C6rSkmcz6ZXLMa7YjtAn2D', -- Meteora
        'PhoeNiXZ8ByJGLkxNfZRnkUfjvmuYqLR89jjFHGqdXY', -- Phoenix
        '2wT8Yq49kHgDzXuPxZSaeLaH1qbmGXtEyPy64bL7aD3c', -- Lifinity V2
        '9xQeWvG816bUx9EPjHmaT23yvVM2ZWbrrpZb9PusVFin', -- Serum DEX v3 / OpenBook
        'SSwpkEEcbUqx4vtoEByFjSkhKdCT862DNVb52nZg1UZ'  -- Saber
     )
    ) AS dex_hop_count,
    -- 3b. DEBT HUB COUNT (Lending / Perps)
    (SELECT COUNT(DISTINCT a.pubkey) FROM UNNEST(accounts) a 
     WHERE a.pubkey IN (
        '6LtLpnUFNByNXLyCoK9wA2MykKAmQNZKBdY8s47dehDc', -- Kamino
        'So1endDq2YkqTCqM3vYpe61CPLUD6nHRN4um8HEufXq', -- Solend
        'MFv2hWf31Z9kbCa1snEPYpcwiNcNVt7BvR9LpXkXVwN', -- MarginFi
        'dRiftyHA39MWEi3m9aunc5MzRF1JYuBsEX6TUqavNbc', -- Drift
        '4MangoMjqJ2firMokCjjGgoK8d4MXcrgL7XJaL3w6fVg', -- Mango V4
        'mv3ekLzLbnVPNxjZApeBp42ewPtzy6H8N15Kq6n2Y1z', -- Mango V3
        'ZETAxsqBRek56DhiGXrn75yj2NHU3aYUnxvHXpkf3aD'  -- Zeta
     )
    ) AS debt_hop_count,
    -- 4. FINANCIAL IMPACT (Max Token Extraction in Transaction)
    -- MAX across all accounts (not instigator-only): profits route through PDAs.
    -- MAX not SUM: avoids cross-token unit mismatch. NULLs (SOL-only txns) -> 0 in preprocessing.
    (SELECT MAX(SAFE_DIVIDE(CAST(post.amount AS FLOAT64) - CAST(pre.amount AS FLOAT64), POW(10, CAST(pre.decimals AS INT64))))
     FROM UNNEST(post_token_balances) post
     INNER JOIN UNNEST(pre_token_balances) pre ON post.account_index = pre.account_index AND post.mint = pre.mint
     WHERE CAST(post.amount AS FLOAT64) > CAST(pre.amount AS FLOAT64)
    ) AS instigator_net_token_profit,
    -- 4b. VALUE EXTRACTION TRIAD
    -- SOL delta: positive = bot gained SOL (native asset / gas)
    COALESCE((
      SELECT (bc.after - bc.before)
      FROM UNNEST(balance_changes) bc
      WHERE bc.account = wallet
      LIMIT 1
    ), 0) / 1e9 AS instigator_sol_delta,
    -- Fiat delta: net fiat-pegged stablecoin flow to instigator wallet
    -- Covers USDC + USDT + PYUSD + USDG: transactional stablecoins present in DeFi lending pools
    COALESCE((
      SELECT SUM(SAFE_DIVIDE(
        CAST(post.amount AS FLOAT64) - CAST(pre.amount AS FLOAT64),
        POW(10, CAST(pre.decimals AS INT64))
      ))
      FROM UNNEST(post_token_balances) post
      INNER JOIN UNNEST(pre_token_balances) pre ON post.account_index = pre.account_index AND post.mint = pre.mint
      WHERE post.owner = wallet
        AND post.mint IN (
          'EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v', -- USDC
          'Es9vMFrzaCERmJfrF4H2FYD4KCoNkY11McCe8BenwNYB', -- USDT
          '2b1kV6DkPAnxd5ixfnxCpjxmKwqjjaYmCZfHsFu24GXo', -- PYUSD (PayPal USD)
          '2u1tszSeqZ3qBWF3uNGPFc8TzMk2tdiwknnRMWGWjGWH'  -- USDG (Global Dollar, Paxos)
        )
    ), 0) AS instigator_fiat_delta,
    -- 5. STRUCTURAL GENERIC FEATURES
    (SELECT COUNT(DISTINCT a.pubkey) FROM UNNEST(accounts) a 
     WHERE a.pubkey NOT IN ('TokenkegQfeZyiNwAJbNbGKPFXCWuBvf9Ss623VQ5DA','11111111111111111111111111111111','ComputeBudget111111111111111111111111111111')
     AND NOT a.signer) AS unique_nonsigner_account_count,
    ARRAY_LENGTH(log_messages) AS log_count,
    ARRAY_LENGTH(accounts) AS num_accounts,
    ARRAY_LENGTH(balance_changes) AS num_balance_changes,
    COALESCE((SELECT MAX(ABS(bc.after - bc.before)) FROM UNNEST(balance_changes) bc), 0)/1e9 AS max_sol_change,
    (SELECT COUNT(DISTINCT tb.mint) FROM UNNEST(pre_token_balances) tb) AS mint_diversity,
    -- 6. PURE COMPOSABILITY: actual unique smart contracts invoked
    -- Parsed from runtime logs: every program call emits 'Program <ADDR> invoke [N]'
    -- Counts known + unknown protocols -- the definitive composability signal
    (SELECT COUNT(DISTINCT REGEXP_EXTRACT(msg, r'Program (\\S+) invoke'))
     FROM UNNEST(log_messages) msg
     WHERE REGEXP_CONTAINS(msg, r'Program \\S+ invoke')
    ) AS unique_program_count
FROM (
    SELECT * FROM stratified_sample
    WHERE wallet IS NOT NULL
    {limit_clause}
)
"""


In [65]:

def downcast_df(df):
    """Reduce RAM usage by downcasting numeric types."""
    fcols = df.select_dtypes('float').columns
    icols = df.select_dtypes('integer').columns
    df[fcols] = df[fcols].apply(pd.to_numeric, downcast='float')
    df[icols] = df[icols].apply(pd.to_numeric, downcast='integer')
    return df

In [66]:
# Assuming client is instantiated elsewhere in your real script
# client = bigquery.Client()

def run_validation(client):
    # ---- Pre-live validation + optional live run ----

    # Step 0: ensure dataset exists in us-central1
    dataset_ref = bigquery.DatasetReference(client.project, DATASET_ID)
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = 'us-central1'
    try:
        client.create_dataset(dataset, exists_ok=True)
        print('Dataset ready.')
    except gcp_exc.Forbidden:
        try:
            client.get_dataset(dataset_ref)
            print('Dataset already exists - continuing.')
        except Exception as exc:
            raise RuntimeError(
                f"Dataset '{DATASET_ID}' missing and service account cannot create it. "
                "Create it manually with location us-central1."
            ) from exc
    except gcp_exc.Conflict:
         print('Dataset already exists - continuing.')

    # Step 1: probe source table access
    print("\nProbing access to public Solana transactions table...")
    probe_sql = """
        SELECT COUNT(*) AS n
        FROM `bigquery-public-data.crypto_solana_mainnet_us.Transactions`
        WHERE block_timestamp BETWEEN '2024-09-01' AND '2024-09-01 00:01:00'
    """
    probe_job = client.query(probe_sql, location='us-central1')
    probe = probe_job.result()
    count = next(iter(probe))['n']
    print(f"  Access confirmed - {count:,} rows in first minute of 2024-09-01 (Job ID: {probe_job.job_id})\n")

    # Step 1.5: feature-level sample validation
    print('Running feature-level sample check before live run...')
    sample_start = '2024-09-01'
    sample_end = '2024-09-01'

    # The new query is a single, triggered CTE pipeline, not a two-step random sampler.
    # Therefore, 'presample_pct' is removed. We just increase the daily_cap limit.
    attempts = [
        {'daily_cap': 20},
        {'daily_cap': 100},
        {'daily_cap': 300},
    ]

    sample_q = f"""
        SELECT *
        FROM {DATASET_ID}.flash_loan_cascade_features
        WHERE tx_date BETWEEN DATE('{sample_start}') AND DATE('{sample_end}')
        ORDER BY block_timestamp, signature
        LIMIT 10
    """

    sample_df = None
    for i, cfg in enumerate(attempts, start=1):
        print(
            f"  Sample attempt {i}/{len(attempts)} "
            f"(daily_cap={cfg['daily_cap']})..."
        )
        
        # Execute the new single-stage Cascade query directly
        build_job = client.query(
            get_cascade_query(
                DATASET_ID,
                sample_start,
                sample_end,
                daily_cap=cfg['daily_cap']
            ),
            location='us-central1',
        )
        build_job.result()
        print(f"    -> Build Job ID: {build_job.job_id}")
        
        extract_job = client.query(sample_q, location='us-central1')
        sample_df = extract_job.to_dataframe()
        print(f"    -> Extract Job ID: {extract_job.job_id}")
        if not sample_df.empty:
            break

    if sample_df is None or sample_df.empty:
        raise RuntimeError(
            'Feature sample query returned 0 rows. '
            'This means no transactions met the Cascade trigger condition (Compute > 500k & 2+ Protocols) on this date. '
            'Try widening sample dates.'
        )

    print("\n--- Raw Data Validation ---")
    print(f"Raw rows: {len(sample_df)}")
    null_counts = sample_df.isnull().sum()
    if null_counts.sum() > 0:
        print("Null counts per column (Raw):")
        print(null_counts[null_counts > 0].to_string())
    else:
        print("No nulls found in raw data.")
    print("---------------------------\n")

    # Apply dtype compression (same as live run), then engineer features for preview
    sample_df = downcast_df(sample_df)
    #sample_df = preprocess.engineer_features(sample_df)

    print(f"Feature sample returned {len(sample_df)} rows.")
    print(f"Feature column count (after engineering): {len(sample_df.columns)}")
    print('BQ columns:')
    print(', '.join(EXPECTED_FEATURE_COLUMNS))

    missing = [c for c in EXPECTED_FEATURE_COLUMNS if c not in sample_df.columns]
    if missing:
        raise RuntimeError('Feature schema mismatch. Missing expected columns: ' + ', '.join(missing))

    print('Schema check passed: all expected feature columns are present.')
    print('\nPreview row:')
    print(sample_df.head(1).to_string(index=False))
    print('\n' + '=' * 65 + '\n')
    
    key_cols = [
      'wallet', 'fee_sol',
      'instigator_sol_delta', 'instigator_fiat_delta', 'instigator_net_token_profit',
      'dex_hop_count', 'debt_hop_count', 'unique_program_count',
      'max_cpi_depth', 'inner_instruction_count',
      'max_sol_change', 'mint_diversity',
    ]                                                                                                                                    
    pd.set_option('display.max_columns', None)                                                                                                        
    pd.set_option('display.width', 300)
    pd.set_option('display.float_format', '{:.6f}'.format)
    print('\nFull sample (all 10 rows):')                                                                                                             
    print(sample_df[key_cols].to_string(index=False))

if __name__ == '__main__':
    client = bigquery.Client(project="analog-subset-491623-j7", credentials=credentials)
    run_validation(client)

Dataset ready.

Probing access to public Solana transactions table...
  Access confirmed - 209,592 rows in first minute of 2024-09-01 (Job ID: 18ab53e0-7b62-4318-9438-cabdd1577117)

Running feature-level sample check before live run...
  Sample attempt 1/3 (daily_cap=20)...
    -> Build Job ID: 58b6314a-b964-4c57-a6a0-0a017b7bea81
    -> Extract Job ID: 83ead1a7-c86b-4f77-b687-af4d4db1aa9f

--- Raw Data Validation ---
Raw rows: 10
Null counts per column (Raw):
instigator_net_token_profit    2
---------------------------

Feature sample returned 10 rows.
Feature column count (after engineering): 22
BQ columns:
signature, block_timestamp, tx_date, wallet, fee_sol, compute_units_consumed, success_flag, num_accounts, log_count, num_balance_changes, max_sol_change, dex_hop_count, debt_hop_count, unique_nonsigner_account_count, involves_sysvar_flag, max_cpi_depth, inner_instruction_count, unique_program_count, instigator_net_token_profit, instigator_sol_delta, instigator_fiat_delta, mint_div

In [ ]:

RUN_LIVE = True  # Set to True when you want chunk export.
if RUN_LIVE:
    save_dir = '/mnt/project_chunks'
    os.makedirs(save_dir, exist_ok=True)
    for s_str, e_str in chunks:
        out_path = os.path.join(save_dir, f'{s_str}.feather')
        if os.path.exists(out_path):
            print(f'[{s_str}] Skip - exists')
            continue
        print(f'\n=== {s_str} ===')
        # 1. Build features for this day
        print('  [1/2] Engineering Cascade features...')
        job1 = client.query(get_cascade_query(DATASET_ID, s_str, e_str, daily_cap=None), location='us-central1')
        job1.result()
        # 2. Extract and downcast
        print('  [2/2] Exporting to feather...')
        q = f"""
            SELECT *
            FROM {DATASET_ID}.flash_loan_cascade_features
            WHERE tx_date = '{s_str}'
        """
        job2 = client.query(q, location='us-central1')
        df = job2.to_dataframe(progress_bar_type='tqdm')
        if not df.empty:
            df = downcast_df(df)
            df.to_feather(out_path)
        bytes_billed = (job1.total_bytes_billed or 0) + (job2.total_bytes_billed or 0)
        total_gb = bytes_billed / (1024**3)
        print(f'      Rows: {len(df):,}')
        print(f'      Data Scanned: {total_gb:.2f} GB')
        print(f'      Est. Cost: ${(total_gb/1024) * 5.0:.4f}')
        del df
        time.sleep(1)
    print('\nAll chunks complete.')
else:
    print('RUN_LIVE=False, so only pre-live validation was executed.')



=== 2024-09-01 ===
  [1/2] Engineering Cascade features...
  [2/2] Exporting to feather...
Job ID 3d6e8cac-b527-4f66-824a-858c0d00ae9f successfully executed: 100%|████████████████████████████████████████████████████████████████████|
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
      Rows: 985,633
      Data Scanned: 369.09 GB
      Est. Cost: $1.8022

=== 2024-09-02 ===
  [1/2] Engineering Cascade features...
  [2/2] Exporting to feather...
Job ID 8d0817ec-a712-40c7-a23e-a3dbccab1341 successfully executed: 100%|████████████████████████████████████████████████████████████████████|
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
      Rows: 1,167,295
      Data Scanned: 398.47 GB
      Est. Cost: $1.9457

=== 2024-09-03 ===
  [1/2] Engineering Cascade features...
  [2/2] Exporting to feather...
Job 

  [2/2] Exporting to feather...
Job ID 7fd1293e-1a87-4134-b321-7f3b0c1be29f successfully executed: 100%|████████████████████████████████████████████████████████████████████|
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
      Rows: 1,456,168
      Data Scanned: 398.09 GB
      Est. Cost: $1.9438

=== 2024-09-19 ===
  [1/2] Engineering Cascade features...
  [2/2] Exporting to feather...
Job ID 2f58adff-b4b0-44bb-b435-3b8e25fcadbb successfully executed: 100%|████████████████████████████████████████████████████████████████████|
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
      Rows: 1,517,493
      Data Scanned: 391.50 GB
      Est. Cost: $1.9116

=== 2024-09-20 ===
  [1/2] Engineering Cascade features...
  [2/2] Exporting to feather...
Job ID f93d6b9b-6387-4bba-a1c5-4978253c9739 successfully execu

Job ID ee66e7bc-819f-4ec8-b995-4a8959a0a10f successfully executed: 100%|████████████████████████████████████████████████████████████████████|
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
      Rows: 1,365,441
      Data Scanned: 425.67 GB
      Est. Cost: $2.0785

=== 2024-10-06 ===
  [1/2] Engineering Cascade features...
  [2/2] Exporting to feather...
Job ID 98cdf715-d3f5-4820-8860-e2c5dbfe3dbd successfully executed: 100%|████████████████████████████████████████████████████████████████████|
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|
      Rows: 1,488,859
      Data Scanned: 405.46 GB
      Est. Cost: $1.9798

=== 2024-10-07 ===
  [1/2] Engineering Cascade features...
